In [ ]:
import pandas as pd
import io
from datetime import datetime, timedelta

def analyze_staffing(path):
    df = pd.read_csv(io.StringIO(path))
    
    df['time_start'] = pd.to_datetime(df['time_start'], format='%H:%M:%S.%f')
    df['time_end'] = pd.to_datetime(df['time_end'], format='%H:%M:%S.%f')
    
    rows = []
    for _, row in df.iterrows():
        start = row['time_start']
        end = row['time_end']
        state = row['new_state']
        
        while start.hour != end.hour and start < end:
            next_hour = (start.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1))
            duration = (next_hour - start).total_seconds()
            rows.append({'hour': start.hour, 'state': state, 'duration': duration})
            start = next_hour
            
        duration = (end - start).total_seconds()
        if duration > 0:
            rows.append({'hour': start.hour, 'state': state, 'duration': duration})

    processed_df = pd.DataFrame(rows)

    overall = processed_df.groupby('state')['duration'].sum()
    overall_total = overall.sum()
    overall_pct = (overall / overall_total * 100).round(2)
    
    print("--- OVERALL PERCENTAGES ---")
    print(overall_pct.to_string(formatters={lambda x: f"{x}%"}))
    print("\n")

    hourly_stats = processed_df.groupby(['hour', 'state'])['duration'].sum().unstack(fill_value=0)
    hourly_total = hourly_stats.sum(axis=1)
    hourly_pct = hourly_stats.divide(hourly_total, axis=0) * 100
    
    print("--- HOURLY PERCENTAGES ---")
    print(hourly_pct.round(2).to_string())

# --- Example Usage ---
path = "output/counter_staffing_20260301.csv"

analyze_staffing(path)

--- OVERALL PERCENTAGES ---


TypeError: Series.to_string() got an unexpected keyword argument 'formatters'